# Session 3 — Meet Spark (part 1 of 2)

In session 2 we cleaned 625 rows with plain Python and calculated that a billion rows would take hours on one CPU core.
Today we meet **Apache Spark**, the engine that big companies use to process data on many machines at once —
and that you will use in the distributed processing course after this one.

We take it slowly. Spark is new for everyone, also for those who already know Python.

**Today you will learn**
1. What Spark is and why data engineers use it
2. How Spark works: driver, executors, partitions
3. How to start Spark: the `SparkSession`
4. Your first DataFrame: `show`, `printSchema`, `count`
5. Columns and functions: `select`, `F.col`, `withColumn`, `F.round`, `F.upper`, `F.lit`, ...
6. Reading a CSV file

**Next session (part 2):** schemas, filtering and sorting, lazy evaluation, and finding data problems with Spark.

## Homework review (15 min)
Open `homework/hw02_python_files.ipynb`.

---
## Setup
Run the next cell. It finds the course data (or downloads it in Colab).

In [ ]:
# Run me first: finds the course data, or downloads it when running in Google Colab.
import subprocess
from pathlib import Path

COURSE_REPO = "https://github.com/alinafazliu/data-engineering-modul-1.git"


def find_data():
    for folder in [Path.cwd(), *Path.cwd().parents, Path("/content/course")]:
        for candidate in [folder / "data", folder / "month_course" / "data"]:
            if (candidate / "reference" / "tracks.csv").exists():
                return candidate


DATA = find_data()
if DATA is None:
    subprocess.run(["git", "clone", "--depth", "1", "--quiet", COURSE_REPO, "/content/course"], check=True)
    DATA = find_data()
print("Course data:", DATA)

---
## Part 1 — What is Spark?

**The problem.** At the end of session 2 we timed our cleaning code: a few microseconds per row. That sounds fast, but Wavely
has about a billion plays per day. On one CPU core that is hours of work — and the data doesn't even fit in one computer's memory.

**The idea: split the work.** Think of counting the votes of an election. One person counting a million ballots needs weeks.
Split the ballots into 100 piles, give each pile to a helper, and let a coordinator add up the 100 results: done in an afternoon.

Spark does exactly this with data:

| Election | Spark |
|---|---|
| the piles of ballots | **partitions**: the data split into pieces |
| the helpers | **executors**: they process the partitions, all at the same time |
| the coordinator | the **driver**: plans the work and collects the results |

**What Spark is**
- An open-source engine for processing large amounts of data on many computers (a **cluster**).
- Started in 2009 at UC Berkeley; today used by companies like Netflix, Uber and Apple, and in cloud platforms
  like Databricks, AWS EMR, Google Dataproc and Microsoft Fabric.
- **PySpark** is Spark for Python: you write Python, Spark does the heavy work.
  Spark itself runs on Java, which is why it needs a few seconds to start.

```
                       ┌──────────── your code (the notebook) ────────────┐
                       │                   DRIVER                          │
                       │  plans the work, splits the data into partitions  │
                       └───────┬──────────────────┬───────────────────┬────┘
                               ▼                  ▼                   ▼
                         ┌──────────┐       ┌──────────┐        ┌──────────┐
                         │ EXECUTOR │       │ EXECUTOR │        │ EXECUTOR │   (separate machines
                         │ part 1,4 │       │ part 2,5 │        │ part 3,6 │    in a real cluster)
                         └──────────┘       └──────────┘        └──────────┘
```

In this course Spark runs in **local mode**: driver and executors share one machine (your laptop or the Colab computer)
and use all of its CPU cores. The code is **exactly the same** on a 100-machine cluster. That is the point.

---
## Part 2 — Starting Spark: the SparkSession

Every Spark program starts by creating a **SparkSession**: your connection to Spark. Run the next cell.
The first time it takes a few seconds (up to 30 in Colab). Then we read it line by line.

In [ ]:
# Start Spark. Colab already has PySpark; anywhere else we install it first.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pyspark>=3.5,<3.6"], check=True)
os.environ["PYSPARK_PYTHON"] = sys.executable  # Spark's workers must use the same Python as the notebook

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("wavely")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.ansi.enabled", "false")  # same behaviour on Spark 3.5 and 4.x: bad casts give null
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready")

What did that cell do?

| Code | What it means |
|---|---|
| `if importlib.util.find_spec("pyspark") is None: ... pip install ...` | Colab already has PySpark; anywhere else we install it once |
| `os.environ["PYSPARK_PYTHON"] = sys.executable` | Spark's workers use the same Python as this notebook |
| `from pyspark.sql import SparkSession` | the tool we need to start Spark |
| `from pyspark.sql import functions as F` | Spark's built-in functions, under the short name `F` (Part 4) |
| `SparkSession.builder` | start describing the Spark we want |
| `.appName("wavely")` | a name for our program, shown in Spark's monitoring page |
| `.master("local[*]")` | **where** to run: on this machine (`local`), with all CPU cores (`*`). On a cluster this points to the cluster |
| `.config(...)` | settings, so everybody gets the same results. You don't need to remember them |
| `.getOrCreate()` | start Spark — or, if it is already running, give us the running one |
| `spark.sparkContext.setLogLevel("ERROR")` | only show real errors, not hundreds of info messages |

The result is stored in the variable **`spark`**. Everything we do with Spark starts from it.

Writing `( ... )` with one `.something()` per line is called **method chaining**: each step gives back an object, and
we call the next method on it. You will see this style everywhere in Spark.

In [ ]:
print(spark.version)                          # which Spark version
print(spark.sparkContext.defaultParallelism)  # how many tasks can run at the same time (= CPU cores here)

same_spark = SparkSession.builder.getOrCreate()
print(same_spark is spark)                    # True: getOrCreate() gives us the session that is already running

In Colab you usually get **2** cores, a laptop has 4 to 16, and a cluster can have thousands. Your code stays the same.

### Exercise 1 (Core) — Your Spark
Use an f-string to print one sentence with the Spark version and the number of cores, like the expected output.

In [ ]:
# YOUR CODE HERE

# expected (your numbers may differ): Spark 3.5.1 runs on 2 cores

---
## Part 3 — Your first DataFrame

A **DataFrame** is a table: rows, and columns with a name and a type. It is like our list of dicts from sessions 1 and 2,
but Spark keeps it split into partitions and can process them in parallel.

`spark.createDataFrame(...)` turns a list of dicts into a DataFrame:

In [ ]:
plays = [
    {"user_id": "U0525", "artist": "Dua Lipa", "seconds": 226, "skipped": False},
    {"user_id": "U0277", "artist": "Drake", "seconds": 6, "skipped": True},
    {"user_id": "U0458", "artist": "Coldplay", "seconds": 280, "skipped": False},
]

df = spark.createDataFrame(plays)
df

Typing `df` only shows the column names and types — **not the data**. To see the rows, ask Spark with `show()`:

In [ ]:
df.show()

`show()` prints the first rows as a table (20 by default; `df.show(5)` shows 5).

The columns are in **alphabetical order** (`artist`, `seconds`, `skipped`, `user_id`), not the order we typed them:
Spark sorts the keys of dicts. In Part 4 you will choose your own order with `select`.

`printSchema()` shows the **schema**: every column with its type.

In [ ]:
df.printSchema()

- Spark guessed the types: `string` (text), `long` (a big whole number) and `boolean` (True/False).
- `nullable = true` means the column may contain **null**: Spark's word for a missing value (Python's `None`).

Two more ways to look at a DataFrame:

In [ ]:
print(df.count(), "rows")
print(df.columns)

| Python list of dicts | Spark DataFrame |
|---|---|
| `print(plays)` | `df.show()` |
| `len(plays)` | `df.count()` |
| `plays[0].keys()` | `df.columns` |
| `type(plays[0]["seconds"])` | `df.printSchema()` |

> **Under the hood:** `df.rdd.getNumPartitions()` tells you how many pieces the data is split into. Try it!

### Exercise 2 (Core) — Your own DataFrame
Complete `my_plays_list` with **3 more** plays (any users, tracks and devices you like), turn it into a DataFrame
`my_plays`, then show it, print its schema and count the rows.

In [ ]:
# YOUR CODE HERE
my_plays_list = [
    {"user_id": "U0101", "track": "Levitating", "seconds": 203, "device": "mobile"},
    # add 3 more plays here
]

my_plays.show()
my_plays.printSchema()
print(my_plays.count(), "rows")  # expected: 4 rows

---
## Part 4 — Columns and functions

### Picking columns: `select`
`select` keeps only the columns you name, in the order you name them:

In [ ]:
df.select("user_id", "artist", "seconds").show()

### A column in Spark: `F.col`
To **calculate** with a column we need a column object: `F.col("seconds")`. It does not contain the values — it is a
description: *"the column called seconds"*. Spark fills in the values for every row when it runs.

In [ ]:
print(F.col("seconds"))
print(F.col("seconds") / 60)

We can use such a column inside `select`. `.alias("minutes")` gives the new column a name
(without it, Spark would call it `(seconds / 60)`):

In [ ]:
df.select(
    "artist",
    "seconds",
    (F.col("seconds") / 60).alias("minutes"),
).show()

The calculation runs for **every row** — no `for` loop needed. That is the big difference to session 2.

### Adding a column: `withColumn`
`withColumn("name", column)` keeps all columns and adds one more (or replaces a column with the same name):

In [ ]:
df_minutes = df.withColumn("minutes", F.col("seconds") / 60)
df_minutes.show()

print(df.columns)          # df is unchanged!
print(df_minutes.columns)

A DataFrame **never changes**. Every step gives you a **new** DataFrame, so store it in a variable if you want to keep it.

### Built-in functions: `F.something`
Spark has hundreds of functions for columns. They work like the Python you know, but on a whole column at once:

| Python (one value) | Spark (a whole column) |
|---|---|
| `round(x, 1)` | `F.round(F.col("x"), 1)` |
| `text.upper()` / `text.lower()` | `F.upper(F.col("text"))` / `F.lower(F.col("text"))` |
| `text.strip()` | `F.trim(F.col("text"))` |
| `len(text)` | `F.length(F.col("text"))` |
| a fixed value, e.g. `"Wavely"` | `F.lit("Wavely")` — the same value in every row |

In [ ]:
(
    df
    .withColumn("minutes", F.round(F.col("seconds") / 60, 1))
    .withColumn("artist_upper", F.upper(F.col("artist")))
    .withColumn("name_length", F.length(F.col("artist")))
    .withColumn("service", F.lit("Wavely"))
    .show()
)

Wrapping the steps in `( )` lets us write one step per line — much easier to read.

Why `F.lit`? `withColumn` needs a **column**, not a plain value. `df.withColumn("service", "Wavely")` gives the error
`Argument col should be a Column, got str`. `F.lit("Wavely")` turns the value into a column.

Functions can be **nested**, like in Python: the inner one runs first. Here is `clean_name` from session 2, in Spark:

In [ ]:
names = spark.createDataFrame([{"name": "  dua LIPA "}, {"name": "DRAKE"}, {"name": "coldplay  "}])

names.withColumn("clean", F.upper(F.trim(F.col("name")))).show()

### Exercise 3 (Core) — Milliseconds
From `df`, select `user_id`, `artist` and a new column `ms` (the seconds × 1000). Store the result in `df_ms`.

```
+-------+--------+------+
|user_id|  artist|    ms|
+-------+--------+------+
|  U0525|Dua Lipa|226000|
|  U0277|   Drake|  6000|
|  U0458|Coldplay|280000|
+-------+--------+------+
```

In [ ]:
# YOUR CODE HERE

df_ms.show()

### Exercise 4 (Core) — More columns
Starting from `df`, add these columns with `withColumn`, then show only `artist`, `artist_lower`, `artist_length`, `minutes`, `country`.
Store the result in `df_more`.
- `artist_lower`: the artist in lower case
- `artist_length`: the number of characters in the artist's name
- `minutes`: seconds / 60, rounded to 1 decimal
- `country`: `"AT"` in every row

```
+--------+------------+-------------+-------+-------+
|  artist|artist_lower|artist_length|minutes|country|
+--------+------------+-------------+-------+-------+
|Dua Lipa|    dua lipa|            8|    3.8|     AT|
|   Drake|       drake|            5|    0.1|     AT|
|Coldplay|    coldplay|            8|    4.7|     AT|
+--------+------------+-------------+-------+-------+
```

In [ ]:
# YOUR CODE HERE

df_more.show()

**Stretch:** add a column `label` like `Dua Lipa (226 s)`. `F.concat(col1, col2, ...)` glues columns together
— use `F.lit` for the fixed text parts.

In [ ]:
# YOUR CODE HERE

---
## Part 5 — Reading a CSV file

Real data comes from files. `spark.read.csv` reads a CSV file into a DataFrame:

- `header=True`: the first line contains the column names
- `inferSchema=True`: Spark looks at the data and **guesses** the types
- Spark wants the path as text, so we write `str(DATA / ...)`

In [ ]:
tracks = spark.read.csv(str(DATA / "reference" / "tracks.csv"), header=True, inferSchema=True)

tracks.show(5)
tracks.printSchema()
print(tracks.count(), "tracks")  # expected: 186 tracks

What happens without `inferSchema=True`? Every column is a `string`, because a CSV file is just text (remember homework 2?):

In [ ]:
spark.read.csv(str(DATA / "reference" / "tracks.csv"), header=True).printSchema()

Guessing is handy for exploring, but slow (Spark reads the file twice) and sometimes wrong.
Next session you learn how data engineers tell Spark the types themselves, with a **schema**.

### Exercise 5 (Core) — Track lengths
From `tracks`, build `track_minutes` with the columns `title`, `genre` and `minutes` (`duration_ms` / 60000, rounded to 1 decimal).
Show 5 rows and count the rows.

```
+----------------+-----+-------+
|           title|genre|minutes|
+----------------+-----+-------+
|Electric Mirrors|  pop|    4.0|
|Electric Skyline|  pop|    3.3|
|Endless Promises|  pop|    3.3|
|   Golden Dreams|  pop|    3.2|
|    Little Ocean|  pop|    3.1|
+----------------+-----+-------+
```

In [ ]:
# YOUR CODE HERE

track_minutes.show(5)
print(track_minutes.count(), "rows")  # expected: 186 rows

**Stretch:** read `DATA / "reference" / "artists.csv"` into `artists` (with `header=True, inferSchema=True`).
Show `artist_name`, `country`, the `genre` in upper case and a column `name_length`.

In [ ]:
# YOUR CODE HERE

---
## Wrap-up

| Concept | Example |
|---|---|
| start Spark | `SparkSession.builder.appName("wavely").master("local[*]").getOrCreate()` |
| create a DataFrame | `spark.createDataFrame(list_of_dicts)` |
| look at it | `show()`, `printSchema()`, `count()`, `columns` |
| pick columns | `df.select("a", "b")` |
| a column | `F.col("a")` |
| calculate and name | `(F.col("a") / 60).alias("minutes")` |
| add a column | `df.withColumn("minutes", F.col("a") / 60)` |
| functions | `F.round`, `F.upper`, `F.lower`, `F.trim`, `F.length`, `F.lit`, `F.concat` |
| read a CSV | `spark.read.csv(path, header=True, inferSchema=True)` |

**Next session (part 2):** telling Spark the types (schemas), keeping rows with `filter`, sorting, lazy evaluation,
and using Spark to find the problems in the messy listening log.

**Homework:** `homework/hw03_spark_first_steps.ipynb`